# LULC - Train and predict

This notebook trains a random forest model, and then predict LULC.

### Steps: 
1. Train model:
Try one model for all sites. (append all CSVs)
export model dump (python pickle?)
in future we may need to make different models for different regions and year ranges.
train the model using the geomad of the year of the input products.

2. Predict:
per grid tile:
  per year:
    load geomad/indices/elevation etc.
    make using get_gadm (buffered 100m)
    predict
This is as a command. 

In [ ]:
# Reload functions during development
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline


# Scientific core
import numpy as np
import pandas as pd
import boto3

# Visualisation

# Geospatial
import rioxarray  # noqa: F401
import zarr  # noqa: F401

# Machine learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
import joblib

# Local
from ldn.typology import classes, colors
from ldn.utils import PREDICTION_VERSION

In [ ]:
import geopandas as gpd
from shapely import wkt
from io import StringIO

class_attr = "lulc"

# Gather all of the CSVs of training points for many AOIs.
# paths = f"training_data/{PREDICTION_VERSION}/{tile_id}/{year}/samples_*.csv"
# files = glob.glob(paths)

BUCKET = "data.ldn.auspatious.com"

s3 = boto3.client("s3")

# List all training CSVs for this version
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"training_data/{PREDICTION_VERSION}/")
keys = [
    obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith("samples.csv")
]
print(f"Found {len(keys)} CSV files")

# Stream each CSV from S3 into a DataFrame (no s3fs needed)
dfs = []
for key in keys:
    print(f"  Loading s3://{BUCKET}/{key}")
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read().decode("utf-8")
    dfs.append(pd.read_csv(StringIO(body)))

samples = pd.concat(dfs, ignore_index=True)
print(f"Combined training data: {len(samples)} samples from {len(dfs)} files")

# Parse WKT geometry so we can do spatial region assignment
samples["geometry"] = samples["geometry"].apply(wkt.loads)
samples = gpd.GeoDataFrame(samples, geometry="geometry", crs="EPSG:4326")

samples.drop(columns=["outlier"], inplace=True, errors="ignore")
# samples.drop(columns=["geometry"], inplace=True, errors="ignore")

print(samples.head())
print("Per class counts:")
print(samples[class_attr].value_counts().sort_index())

# # Singapore is missing 4/Wetland.

### Visualise class pixel counts per Country

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, len(keys), figsize=(4 * len(keys), 4), sharey=True)
if len(keys) == 1:
    axes = [axes]

for ax, df in zip(axes, dfs):
    counts = df[class_attr].value_counts().sort_index()
    bar_colors = [colors.get(cls, "gray") for cls in counts.index]

    ax.bar(
        counts.index.astype(str),
        counts.values,
        color=bar_colors,
        edgecolor="black",
        linewidth=0.5,
    )
    # ax.set_title(Path(f.replace("samples_", "")).stem, fontsize=15)
    ax.set_xlabel("class")
    ax.set_ylabel("count")

# Shared legend
handles = [
    mpatches.Patch(color=col, label=str(cls), edgecolor="black")
    for cls, col in colors.items()
]
fig.legend(
    handles=handles,
    title="class",
    loc="lower center",
    ncol=len(colors),
    bbox_to_anchor=(0.5, -0.05),
)

plt.tight_layout()
# plt.savefig("class_counts_per_file.png", dpi=150, bbox_inches="tight")
plt.show()

# TODO: Kiribati tile is 100% water. Products original classes reflect this.

In [ ]:
# from shapely.geometry import box

# # Define regions to split models by.
# max_y = 90
# min_y = -90

# split_x_1 = -180
# split_x_2 = -107
# split_x_3 = -40
# # split_x_4 = 90 # Puts Singapore and Timor-Leste in Pacific2
# split_x_4 = 128  # Puts Singapore and Timor-Leste in Africa
# split_x_5 = 180

# regions = {
#     # West to East.
#     "pacific1": box(split_x_1, min_y, split_x_2, max_y),
#     "caribbean": box(split_x_2, min_y, split_x_3, max_y),
#     "africa": box(split_x_3, min_y, split_x_4, max_y),
#     "pacific2": box(split_x_4, min_y, split_x_5, max_y),
# }

# # Build a GeoDataFrame of region boxes and spatial-join to samples
# regions_gdf = gpd.GeoDataFrame(
#     {"region": regions.keys()},
#     geometry=list(regions.values()),
#     crs="EPSG:4326",
# )
# samples = gpd.sjoin(
#     samples, regions_gdf[["region", "geometry"]], how="left", predicate="intersects"
# )
# samples.drop(columns=["index_right"], inplace=True, errors="ignore")

# print("Samples per region:")
# print(samples["region"].value_counts())
# print(f"Samples without a region: {samples['region'].isna().sum()}")

# # TODO: Use this region logic once we split the model per region.
# # TODO: Count classes per region. Many will be missing.

In [ ]:
# # Make different models for time periods.
# years = [year for year in range(2000, 2025)]  # 2000 to 2024
# print(years)

# # TODO: Verify these splits.
# # In our GeoMAD logic we use <= 2012 as a condition to use a buffered year and T1 and T2 data.
# time_periods = {
#     "tm_etm": [year for year in range(2000, 2003)],  # L5 + L7 pre-SLC failure
#     "slc_off": [year for year in range(2003, 2013)],  # L7 SLC-off, L5 aging/gone
#     "oli": [year for year in range(2013, 2025)],  # L8/L9, modern radiometry
# }
# print(time_periods)

In [ ]:
# # TODO: Cross regions and periods
# region_periods = {}

## Again filter for outliers (for the combined data for many countries).

In [ ]:
# TODO here balance concatenated training data by class e.g. if one class has a huge amount of samples (after contatenation).
# TODO: in future think about region-splitting per model.
# TODO: Do a sample distrubution check per class AND country/region.
# This can decide which countries should be included to train the model/model per region.
# Focus on getting all classes e.g. other.

from matplotlib import pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Prep feature matrix
exclude_cols = ["lulc", "geometry"]
feature_cols = [c for c in samples.columns if c not in exclude_cols]

samples["outlier"] = False
samples.loc[samples[feature_cols].isna().any(axis=1), "outlier"] = True

nan_rows = samples["outlier"].sum()
print(f"Rows with NaNs (auto-flagged): {nan_rows}")

valid_mask = ~samples[feature_cols].isna().any(axis=1)
valid_idx = samples.index[valid_mask]

X = samples.loc[valid_idx, feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Visualisation helpers
CLUSTER_CMAP = "tab10"

# def plot_pca(ax, Xc, labels, centers, outlier_mask, class_name, best_score):
#     """PCA scatter — inliers coloured by cluster, outliers as red crosses."""
#     pca      = PCA(n_components=2)
#     X_2d     = pca.fit_transform(Xc)
#     c_2d     = pca.transform(centers)
#     var      = pca.explained_variance_ratio_
#     k        = len(np.unique(labels))
#     inliers  = ~outlier_mask

#     ax.scatter(
#         X_2d[inliers, 0], X_2d[inliers, 1],
#         c=labels[inliers], cmap=CLUSTER_CMAP, vmin=0, vmax=max(k - 1, 1),
#         alpha=0.55, s=25, linewidths=0, label='inlier'
#     )
#     if outlier_mask.any():
#         ax.scatter(
#             X_2d[outlier_mask, 0], X_2d[outlier_mask, 1],
#             c='red', marker='x', s=50, linewidths=1.2, label='outlier'
#         )
#     ax.scatter(
#         c_2d[:, 0], c_2d[:, 1],
#         c='black', marker='*', s=180, zorder=5, label='centroid'
#     )
#     ax.set(
#         xlabel=f'PC1 ({var[0]:.1%})',
#         ylabel=f'PC2 ({var[1]:.1%})',
#         title=f'PCA  |  k={k}  sil={best_score:.3f}'
#     )
#     ax.legend(fontsize=7, markerscale=0.9)


# def plot_heatmap(ax, centers, feature_cols, outlier_count, n):
#     """Centroid heatmap — rows = clusters, cols = features."""
#     im = ax.imshow(centers, aspect='auto', cmap='RdBu_r')
#     ax.set_xticks(range(len(feature_cols)))
#     ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=7)
#     ax.set_yticks(range(len(centers)))
#     ax.set_yticklabels([f'C{i}' for i in range(len(centers))], fontsize=8)
#     plt.colorbar(im, ax=ax, label='Scaled value', pad=0.02)

#     # Annotate each cell with the scaled value
#     for r in range(centers.shape[0]):
#         for c in range(centers.shape[1]):
#             ax.text(c, r, f'{centers[r, c]:.1f}',
#                     ha='center', va='center', fontsize=6,
#                     color='white' if abs(centers[r, c]) > 1.2 else 'black')

#     ax.set_title(f'Centroids  |  outliers={outlier_count} ({100*outlier_count/n:.1f}%)')


# 3. Per-class clustering + outlier flagging + visualisation
threshold = 2.0  # tune: lower = more aggressive flagging

_classes = samples.loc[valid_idx, "lulc"].unique()

for cls in _classes:
    mask = (samples.loc[valid_idx, "lulc"] == cls).values
    idx = valid_idx[mask]
    Xc = X_scaled[mask]
    n = len(Xc)

    if n < 6:
        continue
    k_max = min(5, n // 5)
    if k_max < 2:
        continue

    # Optimal k via silhouette
    best_k, best_score = 2, -1
    for k in range(2, k_max + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(Xc)
        score = silhouette_score(Xc, labels)
        if score > best_score:
            best_k, best_score = k, score

    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(Xc)
    labels = km_final.labels_
    centers = km_final.cluster_centers_

    # Outlier flagging
    outlier_mask = np.zeros(n, dtype=bool)
    for cluster_id in range(best_k):
        in_cluster = labels == cluster_id
        pts = Xc[in_cluster]
        dists = cdist(pts, centers[cluster_id].reshape(1, -1)).flatten()
        median_d = np.median(dists)
        if median_d == 0:
            continue
        flagged = dists > threshold * median_d
        outlier_mask[np.where(in_cluster)[0][flagged]] = True

    samples.loc[idx[outlier_mask], "outlier"] = True

    print(
        f"  {str(cls):30s} | n={n:4d} | k={best_k} | sil={best_score:.3f} "
        f"| outliers={outlier_mask.sum():4d} ({100 * outlier_mask.mean():.1f}%)"
    )

    # # Figure: PCA (left) + heatmap (right)
    # fig = plt.figure(figsize=(14, 4.5))
    # fig.suptitle(f'Class: {cls}  (n={n})', fontsize=12, fontweight='bold', y=1.01)

    # gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 1.6], figure=fig)
    # ax_pca  = fig.add_subplot(gs[0])
    # ax_heat = fig.add_subplot(gs[1])

    # plot_pca(ax_pca, Xc, labels, centers, outlier_mask, cls, best_score)
    # plot_heatmap(ax_heat, centers, feature_cols, outlier_mask.sum(), n)

    # plt.tight_layout()
    # plt.show()

# 4. Summary
print(f"\nTotal points : {len(samples):,}")
print(f"Clean        : {(~samples['outlier']).sum():,}")
print(f"Outliers     : {samples['outlier'].sum():,}")
print("Outliers per class:")
print(samples.groupby("lulc")["outlier"].mean().mul(100).round(1).astype(str) + "%")
samples = samples[~samples["outlier"]]  # Remove outliers
samples.drop(columns=["outlier"], inplace=True)

### Test correlation between features. Exclude >95% correlated features.

In [ ]:
# exclude_cols = ['lulc', 'geometry']
# feature_cols = [c for c in samples.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(samples[c])]

# corr = samples[feature_cols].corr().abs()

# # Upper triangle mask (ignore self-correlation on the diagonal)
# upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

# # Find pairs above 0.95
# high_corr_pairs = [
#     (col, row, upper.loc[row, col])
#     for col in upper.columns
#     for row in upper.index
#     if upper.loc[row, col] > 0.95
# ]

# if high_corr_pairs:
#     print("Highly correlated feature pairs (>0.95):")
#     for a, b, r in sorted(high_corr_pairs, key=lambda x: -x[2]):
#         print(f"  {a:12s} ↔ {b:12s}  r={r:.3f}")

#     # Drop the second feature in each pair (keep first encountered)
#     to_drop = {b for _, b, _ in high_corr_pairs}
#     print(f"\nDropping {len(to_drop)} redundant feature(s): {to_drop}")
#     samples.drop(columns=to_drop, inplace=True, errors='ignore')
# else:
#     print("No feature pairs with correlation > 0.95 — keeping all features.")

# # Heatmap using matplotlib only
# fig, ax = plt.subplots(figsize=(12, 10))
# im = ax.imshow(corr.values, cmap="coolwarm", vmin=0, vmax=1)
# ax.set_xticks(range(len(feature_cols)))
# ax.set_yticks(range(len(feature_cols)))
# ax.set_xticklabels(feature_cols, rotation=45, ha="right", fontsize=7)
# ax.set_yticklabels(feature_cols, fontsize=7)
# for i in range(len(feature_cols)):
#     for j in range(len(feature_cols)):
#         ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=5,
#                 color="white" if corr.values[i, j] > 0.7 else "black")
# plt.colorbar(im, ax=ax, label="Absolute correlation")
# ax.set_title("Feature Correlation (absolute)")
# plt.tight_layout()
# plt.show()

## Train the model

In [ ]:
# Split 70/30 into train/test. Splits the classes into train/test in a representative way.
train_gdf, test_gdf = train_test_split(
    samples, test_size=0.3, stratify=samples[class_attr], random_state=42
)

print(f"Training set class distribution:\n{train_gdf[class_attr].value_counts()}")
print(f"Test set class distribution:\n{test_gdf[class_attr].value_counts()}")
print(train_gdf)

# Define features and target
feature_cols = [
    c
    for c in train_gdf.columns
    if c != class_attr and c != "region" and pd.api.types.is_numeric_dtype(train_gdf[c])
]

# Pass DataFrames (not .values) so scikit-learn records feature names.
# This avoids the "fitted without feature names" warning at prediction time.
X_train = train_gdf[feature_cols]
y_train = train_gdf[class_attr]
X_test = test_gdf[feature_cols]
y_test = test_gdf[class_attr]

print(f"Classes: {np.unique(y_train)}")

classifier = RandomForestClassifier(
    n_estimators=500, class_weight="balanced", random_state=42
)
model = classifier.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Feature importance
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(
    ascending=False
)
print("Feature importances:")
print(importances)
# TODO: Drop noisy features and retrain.

present = np.unique(np.concatenate([y_test, y_pred]))
target_names = [
    k for k, v in sorted(classes.items(), key=lambda x: x[1]) if v in present
]
print(f"Target names: {target_names}")

print(classification_report(y_test, y_pred, target_names=target_names, labels=present))

In [ ]:
# One method of validation is for example to train on all countries in the Caribbean, and then predict for the last one.

# Make training data for Cape Verde, but don't use that to train the model. Just use it to test. The below current validation overpromises accuracy.
# Hold out testing.
# Cross-validation testing.
# Cape Verde is mostly bare land = other. SR is really bright compared to Singapore and Fiji.

# Validate on single year data and change calculations over years.
# E.g. a pixel could flip between grass and crop year to year just due to randomness in the model's training data.
# RF Classification Report - check the confidence map - how many trees voted the same way. 50% confidence is standard. Post-processing can be done e.g. we smooth these temporal flips to one class over time. Post-processing should be minimal.
# Another post-processing method is to smooth any single pixel surrounded in space or time by another class to that same class.

In [ ]:
from pathlib import Path

joblib_path = Path(
    f"../../ldn/models/{PREDICTION_VERSION}/lulc_random_forest_model.joblib"
)
joblib_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, joblib_path)

# Write the model to S3 so it can be loaded in prediction step.
# TODO: Per region/time period model.
s3_key = f"models/{PREDICTION_VERSION}/lulc_random_forest_model.joblib"
s3.upload_file(str(joblib_path), BUCKET, s3_key)

s3_url = f"https://s3.us-west-2.amazonaws.com/{BUCKET}/{s3_key}"
print(f"Uploaded model to {s3_url}")

# # Load the model
# model = joblib.load(joblib_path)